# 마피아42 리플레이 크롤러 (완성본 v4)

이미지는 참가자(players) 목록에 픽당 1번만 저장합니다. 각 채팅 이벤트는 이미지를 따로 담지 않고 pick 번호로만 참조해서 JSON 용량을 크게 줄였습니다.

**사용법**: 셀을 위에서부터 순서대로 실행(Shift+Enter) → 2번 셀에서 URL만 바꾸기

## 1. 설치 (최초 1회, 1~2분)

In [ ]:
!pip install playwright beautifulsoup4 -q
!playwright install chromium --with-deps

## 2. 여기에 리플레이 URL 입력

In [ ]:
REPLAY_URL = "https://mafia42.com/history/kr/191bd6f52ff0d67516f03a5651aa918a"  # <- 실제 링크로 교체

## 3. 크롤링 + 파싱 + 이미지 base64 변환 (전체 로직)

In [ ]:
import re
import json
import base64
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright


def extract_job(img_tag):
    if not img_tag or not img_tag.get("src"):
        return None, None
    src = img_tag["src"]
    m = re.search(r"jobthumb_([a-zA-Z]+)\.png", src)
    job_name = m.group(1) if m else None
    return job_name, src


def parse_player_list(html: str):
    soup = BeautifulSoup(html, "html.parser")
    players = []
    for idx, item in enumerate(soup.select("#user-table .item"), start=1):
        job_img = item.find("img", class_="job-icon-img")
        job_name, job_icon_url = extract_job(job_img)
        nickname_el = item.find("div", class_="nick-name")
        players.append({
            "pick": idx,
            "nickname": nickname_el.get_text(strip=True) if nickname_el else None,
            "job": job_name,
            "job_icon_url": job_icon_url,  # 디버그/대체용, 이미지 본체는 job_icon_data에
        })
    return players


def parse_replay_html(html: str, players):
    """이벤트에는 이미지를 넣지 않는다. pick 번호만 있으면
    players[pick].job_icon_data로 조회 가능하기 때문."""
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("section", class_="table")
    events = []
    nick_to_pick = {p["nickname"]: p["pick"] for p in players if p["nickname"]}
    if table is None:
        return events

    for child in table.find_all(recursive=False):
        classes = child.get("class", []) or []

        if "system" in classes:
            events.append({"event_type": "system", "text": child.get_text(strip=True)})

        elif "chat-data-container" in classes:
            job_img = child.find("img", class_="job-icon-img")
            job_name, _ = extract_job(job_img)  # 이미지 URL은 버리고 직업명만 사용
            nickname_el = child.find("div", class_="nick-name")
            nickname = nickname_el.get_text(strip=True) if nickname_el else None
            pick = nick_to_pick.get(nickname)

            chat_container = child.find("div", class_=lambda c: c and "chat-container" in c)
            if chat_container:
                bubbles = chat_container.find_all("div", class_=lambda c: c and "chat-bubble" in c)
                for bubble in bubbles:
                    b_classes = bubble.get("class", [])
                    chat_type = "CHAT"
                    for t in ["MAFIACHAT", "GHOSTCHAT", "MEGAPHONE", "CHAT"]:
                        if t in b_classes:
                            chat_type = t
                            break
                    events.append({
                        "event_type": "chat", "chat_type": chat_type,
                        "nickname": nickname, "pick": pick, "job": job_name,
                        "message": bubble.get_text(strip=True)
                        # job_icon_data 없음 -> 웹사이트에서 players[pick]로 조회
                    })

        elif "secret-letter-container" in classes:
            to_el = child.find("div", class_="to")
            from_el = child.find("div", class_="from")
            msg_el = child.find("div", class_="message")
            to_name = to_el.get_text(strip=True).replace("To.", "") if to_el else None
            from_name = from_el.get_text(strip=True).replace("from.", "") if from_el else None
            events.append({
                "event_type": "secret_letter",
                "to": to_name, "to_pick": nick_to_pick.get(to_name),
                "from": from_name, "from_pick": nick_to_pick.get(from_name),
                "message": msg_el.get_text(strip=True) if msg_el else None
            })
    return events


async def crawl_and_build(history_url: str):
    """크롤링 + 파싱 + 이미지 base64 변환 (players에만 1회씩 저장)."""
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        api_url_holder = {}

        def handle_request(req):
            if "GetMafiaChat" in req.url or "lambda-url" in req.url:
                api_url_holder["url"] = req.url

        page.on("request", handle_request)

        await page.goto(history_url, wait_until="networkidle", timeout=30000)
        await page.wait_for_timeout(2000)

        api_url = api_url_holder.get("url")
        if not api_url:
            await browser.close()
            raise RuntimeError("GetMafiaChat API 주소를 찾지 못했습니다. URL이 유효한지 확인하세요.")

        await page.goto(api_url, wait_until="networkidle", timeout=30000)
        await page.wait_for_timeout(3000)

        rendered_html = await page.content()

        players = parse_player_list(rendered_html)
        events = parse_replay_html(rendered_html, players)

        # 이미지는 players에만, 픽당 딱 1번씩 변환
        for p_ in players:
            url = p_.get("job_icon_url")
            if not url:
                p_["job_icon_data"] = None
                continue
            try:
                resp = await page.context.request.get(url)
                if resp.ok:
                    body = await resp.body()
                    content_type = resp.headers.get("content-type", "image/png")
                    p_["job_icon_data"] = f"data:{content_type};base64,{base64.b64encode(body).decode()}"
                else:
                    p_["job_icon_data"] = None
            except Exception as e:
                print(f"이미지 변환 실패 ({url}): {e}")
                p_["job_icon_data"] = None

        await browser.close()

    return players, events

## 4. 실행

In [ ]:
players, events = await crawl_and_build(REPLAY_URL)

print(f"참가자 수: {len(players)}")
print(f"총 이벤트 수: {len(events)}")

ok_count = sum(1 for p in players if p.get("job_icon_data"))
print(f"이미지 변환 성공: {ok_count}/{len(players)}")

result = {"source_url": REPLAY_URL, "players": players, "events": events}

with open("parsed_replay.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

import os
size_kb = os.path.getsize("parsed_replay.json") / 1024
print(f"저장 완료: parsed_replay.json ({size_kb:.1f} KB)")

## 5. 결과 미리보기 (선택)

In [ ]:
print("--- 참가자 명단 ---")
for p in players:
    has_img = "✅" if p.get("job_icon_data") else "❌"
    print(f"{p['pick']}. {p['nickname']} ({p['job']}) 이미지:{has_img}")

print("\n--- 앞 5개 이벤트 (이미지 없이 가벼움) ---")
for e in events[:5]:
    print(e)

## 6. 파일 다운로드 (선택)

In [ ]:
from google.colab import files
files.download("parsed_replay.json")